In [1]:
import torch
# vecteur de position 3D
pos = torch.randn(3)
pos

tensor([ 0.0691, -0.1960, -0.6989])

In [2]:
pos

tensor([ 0.0691, -0.1960, -0.6989])

In [3]:
Z_l = Z.tolist()
[f"Z={val}" for val in Z_l]

NameError: name 'Z' is not defined

In [ ]:
import plotly.express as px
import numpy as np
import torch
X = torch.randn(5,3, requires_grad=False)
Z = torch.tensor([6, 8, 1, 1, 1])
pos = X.detach()
fig = px.scatter_3d(
    x = pos[:, 0],
    y = pos[:, 1],
    z = pos[:,2],
    labels={"x": "Position X (Å)", "y": "Position Y (Å)"},
    color = [f"{val}" for val in Z.tolist()]
)

fig.show()

In [5]:
from torch_geometric.datasets import QM9

dataset = QM9(
    root=r"C:\Users\juliu\OneDrive\Bureau\FAEnet\FAENet\data\qm9"
)
molecule = dataset[0]

c:\Users\juliu\OneDrive\Bureau\FAEnet\FAENet\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\juliu\OneDrive\Bureau\FAEnet\FAENet\.venv\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [ ]:
pos = molecule["pos"] #Nx3
import plotly.express as px
Z = molecule["z"]
fig = px.scatter_3d(
    x = pos[:,0],
    y = pos[:,1],
    z = pos[:,2],
    color = Z,
)
fig.show()

In [ ]:
molecule["edge_index"]

tensor([[0, 0, 0, 0, 1, 2, 3, 4],
        [1, 2, 3, 4, 0, 0, 0, 0]])

In [ ]:
molecule

Data(x=[5, 11], edge_index=[2, 8], edge_attr=[8, 4], y=[1, 19], pos=[5, 3], idx=[1], name='gdb_1', z=[5])

In [ ]:
pos = molecule.pos
pos

tensor([[-1.2700e-02,  1.0858e+00,  8.0000e-03],
        [ 2.2000e-03, -6.0000e-03,  2.0000e-03],
        [ 1.0117e+00,  1.4638e+00,  3.0000e-04],
        [-5.4080e-01,  1.4475e+00, -8.7660e-01],
        [-5.2380e-01,  1.4379e+00,  9.0640e-01]])

In [ ]:
# créer un batch
from torch_geometric.data import Batch, Data
L_batch = []
for i in range(32):
    molecule = dataset[i]
    molecule_essentiel = Data(pos = molecule["pos"], z = molecule["z"])
    L_batch.append(molecule_essentiel)

L_batch
B = Batch.from_data_list(L_batch)
B.ptr
# indice de ptr sont les mêmes que les indices de position
# donc on peut slice avec
B.pos[B.ptr[k]:B.ptr[k+1],:]
# batch prend indice de l'élement du batch en entrée
# ça donne la molécule entière pas la ligne du batch
# dans un vrai batch pytorch les inputs ont les mêmes dimensions donc on peut juste avoir B, N, 3
# pas le cas ici

tensor([[ 0.5995,  0.0000,  1.0000],
        [-0.5995,  0.0000,  1.0000],
        [-1.6616,  0.0000,  1.0000],
        [ 1.6616,  0.0000,  1.0000]])

In [ ]:
dataset.z.min()

tensor(1)

In [ ]:
from torch import nn
emb = nn.Embedding(dataset.z.max()+1, 16) # embedding attend un indice commencant à 0, nous il commence à 1, donc on a besoin d'une case en plus
h = emb(B.z)

In [ ]:
emb.weight[1]

tensor([ 1.3298, -0.8153,  0.6351,  0.1343,  0.5343,  1.3162, -0.4667,  1.1496,
         0.3753,  0.1092, -0.0372,  0.4167,  0.4370, -0.9601, -0.5469,  0.5086],
       grad_fn=<SelectBackward0>)

In [ ]:
elements

tensor([1, 6, 7, 8, 9])

In [ ]:
elements = torch.unique(dataset.z, sorted=True)

emb = nn.Embedding(len(elements), 16)
z_to_indice = {z:i for i,z in enumerate(elements.tolist())}
z_to_indice
B.z
#créer un tenseur qui map les z du batch à leurs indices
Batch_z_indices = torch.tensor([z_to_indice[z] for z in B.z.tolist()])
h_z = emb(Batch_z_indices)
emb.weight[0]
indice_to_z = {i:z for z, i in z_to_indice.items()}
indice_to_z
z_to_indice
indice_to_z

{0: 1, 1: 6, 2: 7, 3: 8, 4: 9}

In [11]:
X_global

tensor([[-0.0050,  0.0115, -0.3822],
        [-0.0072,  1.1486,  0.0201],
        [-1.3252,  1.8585,  0.3040],
        [-1.2369,  3.0239,  1.1278],
        [-0.0384,  3.6491,  1.2135],
        [ 1.1032,  3.2767,  0.8264],
        [ 1.2569,  1.9655,  0.2190],
        [-1.7625,  2.1442, -0.6634],
        [-2.0126,  1.1610,  0.7881],
        [-0.1670,  4.6054,  1.7164],
        [ 1.9518,  1.3661,  0.8209],
        [ 1.7459,  2.0710, -0.7593]])

In [27]:
torch.tensor([-1.0, 1.0, -1.0]).view(3,1).expand(3,3)

tensor([[-1., -1., -1.],
        [ 1.,  1.,  1.],
        [-1., -1., -1.]])

In [22]:
def rho1_inv(X_global, t, U):
    return (X_global - t) @ U
# t 1d array so it will broadcast to match the lines of X_global
# U is of size 1x3
molecule = dataset[2003]
X_global = molecule.pos
U = torch.tensor([-1.0, 1.0, -1.0]).expand(3,3)
t = X_global.mean(dim=0)
X_local = rho1_inv(X_global, t, U)
X_local


tensor([[ 2.9604, -2.9604,  2.9604],
        [ 1.4232, -1.4232,  1.4232],
        [ 1.7474, -1.7474,  1.7474],
        [-0.3301,  0.3301, -0.3301],
        [-2.2395,  2.2395, -2.2395],
        [-2.6216,  2.6216, -2.6216],
        [-0.8567,  0.8567, -0.8567],
        [ 2.8664, -2.8664,  2.8664],
        [ 2.6482, -2.6482,  2.6482],
        [-3.5701,  3.5701, -3.5701],
        [-1.5541,  1.5541, -1.5541],
        [-0.4729,  0.4729, -0.4729]])

In [ ]:
def rho1_inv(X_global, t, U):
    return (X_global - t) @ U            # [n, 3] @ [3, 3] -> [n, 3]

X = molecule.pos                         # [n, 3]
t = X.mean(dim=0) # [1,3]

Xc = X-t # broadcasting

In [ ]:
h.size()

torch.Size([230, 16])

In [ ]:
B

DataBatch(pos=[230, 3], z=[230], batch=[230], ptr=[33])

In [31]:
# %%
from pathlib import Path

from mendeleev import element
import torch
from torch import nn
from torch_geometric.datasets import QM9

print(Path.cwd())

# %%

dataset = QM9(root="../data/qm9")

# %%

molecule = dataset[2003]
print("Nom de la molécule :", molecule.name, "\n")
print("Numéros atomiques :\n", molecule.z, "\n") # Z du papier
print("Coordonnées 3D pour chaque atome :\n", molecule.pos) # X du papier

# %% 

print("Caractéristiques des atomes : ", molecule.x)
"""
Colonnes 
- 0 à 4 : type d'atome encodé en one-hot (H, C, N, O, F)
- 5 : Numéro atomique 
- 6 : Nombre de voisins liés chimiquement 
- 7 : Nombre d'hydrogènes liés
- 8 : Charge formelle
- 9 : Nombre d'électrons non appariés 
- 10 : Indique si l'atome est aromatique
"""

# %%

# Dans QM9, chaque liaison chimique est représentée dans les deux sens.
print(molecule.edge_index)
# Par exemple pour la molécule 2003 du dataset, l'atome 0 (O) est lié à l'atome 6 (C)


# %%

print(molecule.edge_attr)
"""
Matrice de vecteurs one-hot qui décrit l'arête située dans la colonne 
correspondante de edge_index.
- (1,0,0,0) : liaison simple
- (0,1,0,0) : liaison double 
- (0,0,1,0) : liaison triple
- (0,0,0,1) : liaison aromatique
"""

print(molecule.y) # regression targets

# %%

def rho1_inv(X_global, t, U_star):
    """
    Fonction pour exprimer X du repère global au repère local 
    """
    return (X_global - t) @ U_star

def rho1(X_local, t, U_star):
    """
    Fonction pour exprimer X du repère local au repère global 
    """
    return X_local@ U_star.t() + t

def SFA(X, k=None):

    t = X.mean(axis=0)
    X_c = X - t 
    Sigma = X_c.t() @ X_c
    _, eigenvectors = torch.linalg.eigh(Sigma)
    signs = torch.cartesian_prod(
        torch.tensor([-1.0, 1.0]),
        torch.tensor([-1.0, 1.0]),
        torch.tensor([-1.0, 1.0])
    ).unsqueeze(1)
    U = signs * eigenvectors
    if k is not None:
        U_star = U[k]
    else:
        U_star = U[torch.randint(0, 7, size=(1,)).item()]
    return U_star

def adjacency_matrix(X_local, cutoff):

    D = torch.cdist(X_local, X_local)
    A = (D < cutoff).float()
    for i in range(X_local.shape[0]):
        A[i,i] = 0.0
    edge_index = torch.nonzero(A).t() # same shape que dans QM9 (2 x E)
    return A, D, edge_index

# %%
X = molecule.pos
t = X.mean(axis=0)
U_star1 = SFA(X,0)
U_star2 = SFA(X,5)

X_local = rho1_inv(X, t, U_star1)
X_global = rho1(X_local, t, U_star1)
assert torch.isclose(X, X_global).all().item() == True # pour voir si rho1 * rho1_inv = Id

# %%
X_local2 = rho1_inv(X, t, U_star2)
A, D, edge_index = adjacency_matrix(X_local, cutoff=6)
_, D2, _ = adjacency_matrix(X_local2, cutoff=6)

# test pour vérifier que la distance est la même entre 2 repères différents
assert torch.isclose(D, D2).all().item() == True

# %%


z_0 = molecule.z[0].item()
atome = element(z_0)
embed_0 = torch.tensor(
    [
        z_0,
        atome.group_id,
        atome.period,
        atome.atomic_radius,
        atome.atomic_volume,
        atome.density,
        atome.dipole_polarizability,
        atome.electron_affinity,
        atome.vdw_radius,
        #atome.metallic_radius,
        atome.covalent_radius,
        atome.ionenergies[1],
        atome.ionenergies[2]
    ]
)

#%%
print(embed_0)


c:\Users\juliu\OneDrive\Bureau\FAEnet\FAENet\sandbox_jules
Nom de la molécule : gdb_2057 

Numéros atomiques :
 tensor([8, 6, 6, 8, 6, 7, 6, 1, 1, 1, 1, 1]) 

Coordonnées 3D pour chaque atome :
 tensor([[-0.0050,  0.0115, -0.3822],
        [-0.0072,  1.1486,  0.0201],
        [-1.3252,  1.8585,  0.3040],
        [-1.2369,  3.0239,  1.1278],
        [-0.0384,  3.6491,  1.2135],
        [ 1.1032,  3.2767,  0.8264],
        [ 1.2569,  1.9655,  0.2190],
        [-1.7625,  2.1442, -0.6634],
        [-2.0126,  1.1610,  0.7881],
        [-0.1670,  4.6054,  1.7164],
        [ 1.9518,  1.3661,  0.8209],
        [ 1.7459,  2.0710, -0.7593]])
Caractéristiques des atomes :  tensor([[0., 0., 0., 1., 0., 8., 0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 6., 0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 6., 0., 0., 0., 0., 2.],
        [0., 0., 0., 1., 0., 8., 0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 6., 0., 0., 0., 0., 1.],
        [0., 0., 1., 0., 0., 7., 0., 0., 0., 0., 0.],
     

In [ ]:

z_0 = molecule.z
atome = element(z_0)
carac_0 = torch.tensor(
    [
        z_0,
        atome.group_id,
        atome.period,
        atome.atomic_radius,
        atome.atomic_volume,
        atome.density,
        atome.dipole_polarizability,
        atome.electron_affinity,
        atome.vdw_radius,
        #atome.metallic_radius,
        atome.covalent_radius,
        atome.ionenergies[1],
        atome.ionenergies[2]
    ]
)
d = 16
#%%
z_values = dataset.z.unique().tolist()
z_to_indices = {z:i for i,z in enumerate(z_values)}
indices_to_z = {i:z for z,i in z_to_indices.items()}
from torch import nn

emb = nn.Embedding(max(z_to_indices.values())+1, d)
embedded_z0 = emb(z_to_indices[z_0])


TypeError: unsupported format string passed to torch._C._TensorMeta.__format__

In [58]:
emb.weight

Parameter containing:
tensor([[-0.6153, -1.2572,  0.7375,  0.0364,  1.6630,  0.7293, -1.3832, -1.5063,
          0.3024,  0.0921,  1.1538,  0.5171,  0.0437,  0.4706,  2.9735, -1.0270],
        [-0.7883,  0.2204, -0.7992,  0.2238, -0.6725, -2.5354, -0.3842, -0.6283,
         -0.1371,  0.1756, -0.6864, -1.4037,  0.0676,  0.2303,  0.1309,  0.2788],
        [-0.6352,  0.8946, -0.4012, -0.2012,  0.2874, -1.5541, -1.0414,  0.7036,
          0.1021, -0.3343, -0.0090,  0.8809,  0.5998,  0.0279, -1.4321, -1.0294],
        [-0.4971, -0.2317,  1.1298, -0.0544,  0.0755,  1.0921, -0.8341,  1.6808,
         -0.7581, -0.8963,  0.5985, -0.7174,  0.4024, -0.9062,  0.8063,  1.0970],
        [-0.6564, -0.2668,  0.0847, -0.9409, -2.5236,  0.6106, -0.7990,  0.8437,
          1.1949,  0.5291,  0.7550, -1.0416, -1.1403, -1.0484, -0.7719,  1.1189]],
       requires_grad=True)

In [56]:
z_0

tensor([8, 6, 6, 8, 6, 7, 6, 1, 1, 1, 1, 1])

In [55]:
int(z_0)

ValueError: only one element tensors can be converted to Python scalars

In [43]:
max(z_to_indices.values())

4